# 05 第三章实验框架

本 Notebook 建立后续实验入口。第一轮重点是让每个方向都有可运行示例，后续可以逐步扩展成专题实验。

当前包含：

* Fourier 逼近周期函数；
* Gibbs 现象的最小示例；
* 自适应分段线性逼近；
* 后续实验目录。


## 为什么单独保留实验框架

前几个 notebook 主要讲确定的理论主线：正交多项式、最小二乘和 Padé。本节则保留若干会在后续扩展成专题的方向。

这些方向有一个共同点：它们都不是单一公式，而是“函数性质、逼近空间和算法选择”之间的配合问题。例如：

* 周期光滑函数适合 Fourier 逼近；
* 不连续函数会暴露 Gibbs 现象；
* 局部变化剧烈的函数适合自适应节点；
* 噪声数据需要正则化或模型选择。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import adaptive_piecewise_linear, piecewise_linear_interpolate


## Fourier 逼近周期函数

三角多项式逼近写作

$$
S_N(x)=a_0+\sum_{k=1}^{N}\left(a_k\cos kx+b_k\sin kx\right).
$$

也可以写成复指数形式

$$
S_N(x)=\sum_{k=-N}^{N}\hat f_k e^{ikx}.
$$

连续理论中，Fourier 系数由积分给出；离散计算中，等距采样值可以通过离散 Fourier 变换转换为频率系数。FFT 是快速计算这个变换的算法。

这里用 `numpy.fft` 做一个最小示例，重点是说明 FFT 服务于三角多项式逼近，而不是孤立的数组技巧。


## DFT 系数的含义

连续 Fourier 系数是积分：

$$
\hat f_k=\frac{1}{2\pi}\int_0^{2\pi} f(x)e^{-ikx}\,dx.
$$

离散 Fourier 变换使用等距采样近似这些系数。若

$$
x_j=\frac{2\pi j}{N},\qquad f_j=f(x_j),
$$

则 DFT 计算的是

$$
\hat f_k\approx \frac{1}{N}\sum_{j=0}^{N-1}f_j e^{-2\pi i jk/N}.
$$

代码中的 `np.fft.rfft(y) / N` 就是在做这个归一化。频谱图中只有频率 3 和 5 附近明显，是因为示例函数本来就是 $\sin(3x)$ 和 $\cos(5x)$ 的组合。


In [ ]:
N = 128
x = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
y = np.sin(3 * x) + 0.4 * np.cos(5 * x)
coefficients = np.fft.rfft(y) / N
frequencies = np.fft.rfftfreq(N, d=1 / N)

plt.figure(figsize=(8, 4))
plt.stem(frequencies[:12], np.abs(coefficients[:12]))
plt.title("周期函数的离散频谱")
plt.xlabel("频率编号")
plt.ylabel("系数幅值")
plt.tight_layout()
plt.show()


## Gibbs 现象入口

对不连续周期函数，Fourier 部分和在跳跃点附近会出现过冲。增加项数会压缩振荡区域，但不会简单消除过冲高度。

这个现象说明：全局光滑基函数在处理不连续结构时会遇到困难。它和多项式全局逼近中的端点振荡、Runge 现象有相似的警示意义：基函数空间必须和目标函数的光滑性匹配。

下面只给出可运行入口，完整推导后续补充。


## Gibbs 现象的理论含义

方波有跳跃间断，而 Fourier 部分和使用的是全局光滑的三角函数。为了逼近跳跃，三角多项式只能在跳跃点附近产生振荡。

增加保留频率数时，振荡区域会变窄，但过冲高度不会按同样方式消失。这个现象提醒我们：

* 光滑周期函数适合 Fourier 逼近；
* 分段光滑但不连续的函数需要特别处理间断；
* 只增加全局基函数数量，不一定能消除局部结构带来的误差。

这和第二章 Runge 现象、本章高次多项式过拟合一样，都是“方法与函数性质不匹配”的表现。


In [ ]:
def square_wave(x):
    return np.where(np.sin(x) >= 0, 1.0, -1.0)

x_grid = np.linspace(-np.pi, np.pi, 1024, endpoint=False)
y_square = square_wave(x_grid)
fft_coefficients = np.fft.fft(y_square)

plt.figure(figsize=(8, 4.5))
plt.plot(x_grid, y_square, color="gray", label="方波")
for keep in [5, 15, 45]:
    filtered = np.zeros_like(fft_coefficients)
    filtered[: keep + 1] = fft_coefficients[: keep + 1]
    filtered[-keep:] = fft_coefficients[-keep:]
    y_partial = np.fft.ifft(filtered).real
    plt.plot(x_grid, y_partial, label=f"保留 {keep} 个频率")
plt.title("Gibbs 现象入口示例")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()


## 自适应分段线性逼近

自适应逼近的思想是：在误差大的地方加密节点，在误差小的地方减少节点。

本节使用最简单的中点误差估计。对区间 $[a,b]$，比较真实中点值

$$
f\left(\frac{a+b}{2}\right)
$$

和线性插值中点值

$$
\frac{f(a)+f(b)}{2}.
$$

如果二者差值超过容差，就把区间二分并继续递归。这个方法不是最高效的自适应算法，但非常适合说明“误差估计驱动网格加密”的基本思想。


## 自适应分段线性算法伪代码

当前自适应算法可以写成：

```text
refine(a, b):
    mid = (a + b) / 2
    error = |f(mid) - (f(a) + f(b)) / 2|
    if error <= tolerance:
        keep interval [a, b]
    else:
        refine(a, mid)
        refine(mid, b)
```

这里的误差估计来自一个简单事实：如果函数在区间上接近线性，那么真实中点值应该接近两端点函数值的平均。偏差越大，说明该区间弯曲越明显，越需要加密。

这只是最基础的自适应思想。更高阶的方法可以使用三次样条误差估计、Chebyshev 系数衰减、或者相邻阶近似之间的差值来判断是否细分。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_adapt, y_adapt = adaptive_piecewise_linear(runge, -1.0, 1.0, tolerance=2e-3, max_depth=12)
x_eval = np.linspace(-1.0, 1.0, 600)
y_eval = piecewise_linear_interpolate(x_adapt, y_adapt, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, runge(x_eval), label="目标函数")
plt.plot(x_eval, y_eval, "--", label="自适应分段线性逼近")
plt.scatter(x_adapt, y_adapt, s=18, color="black", label="自适应节点")
plt.title("自适应节点在曲率较大区域加密")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()

print("自适应节点数:", x_adapt.size)


## 自适应结果分析

Runge 函数

$$
f(x)=\frac{1}{1+25x^2}
$$

在 $x=0$ 附近变化更快，在两端更平缓。自适应算法会在中间区域放置更多节点，这说明节点分布开始跟随函数曲率，而不是机械等距分布。

这个实验和第二章插值中的 Chebyshev 节点有相似思想：自由度不一定要均匀放置。好的节点分布应该服务于误差控制。


## 后续实验目录

后续可以把以下内容补成独立专题：

* Chebyshev 级数逼近不同光滑性函数，观察系数衰减；
* Legendre 投影与 Chebyshev 拟合误差对比；
* 正交多项式最小二乘拟合与幂基拟合对比；
* Taylor 与 Padé 对比更多函数，例如 $\log(1+x)$ 和 $\tan x$；
* Padé 虚假极点检测实验；
* Gibbs 现象的过冲高度与项数关系；
* 自适应三次样条逼近；
* 正则化最小二乘拟合和交叉验证。


## 实验小结

本 Notebook 目前是实验框架，不追求一次性讲完所有理论。它的作用是为后续扩展保留可运行入口，并展示第三章的主要实验方向。

这些实验共同服务于一个核心问题：当目标函数或数据性质发生变化时，应该如何选择逼近空间、误差范数和数值算法。
